# pandas.DataFrame

Знов про категоріальні дані

In [1]:
import numpy as np
import pandas as pd
import sys
print(sys.platform, sys.version, f'numpy version {np.__version__}',  f'pandas version {pd.__version__}',sep='\n')

win32
3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
numpy version 2.4.1
pandas version 3.0.0


In [2]:
from pathlib import Path
data = pd.read_csv(Path('prepare', 'exams.csv'), sep=',', encoding='utf-8', names=['exam','points'],
                  dtype={'points':'Int64'})
data.head(3)

,exam,points
0,exam1,94
1,exam1,80
2,exam1,61


In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 290 entries, 0 to 289
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   exam    290 non-null    str  
 1   points  270 non-null    Int64
dtypes: Int64(1), str(1)
memory usage: 4.9 KB


In [4]:
data.fillna(0, inplace=True)

,exam,points
0,exam1,94
1,exam1,80
2,exam1,61
3,exam1,75
4,exam1,80
...,...,...
285,exam3,69
286,exam3,98
287,exam3,74
288,exam3,81


In [5]:
data.groupby('exam').mean()

,points
exam,
exam1,69.161765
exam2,75.068966
exam3,63.237037


In [6]:
data[data.points<60]

,exam,points
5,exam1,54
8,exam1,22
10,exam1,0
12,exam1,0
13,exam1,0
14,exam1,54
18,exam1,0
20,exam1,26
30,exam1,0
31,exam1,0


Від балів хочемо перейти до категоріальної національної шкали оцінок.

Документація на функцію <code>cut</code>  <a href="https://pandas.pydata.org/docs/reference/api/pandas.cut.html#pandas.cut" target=_blank rel="noopener noreferrer">тут</a>.

Задаємо інтервали. За умовчанням ліва границя не включається, права включається.

In [7]:
bins = [-1, 0, 59, 74, 89, 100]

In [8]:
data['basket'] = pd.cut(data.points, bins, 
            labels=['NA', 'mark2', 'mark3', 'mark4', 'mark5'])
data.head()

,exam,points,basket
0,exam1,94,mark5
1,exam1,80,mark4
2,exam1,61,mark3
3,exam1,75,mark4
4,exam1,80,mark4


In [9]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 290 entries, 0 to 289
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   exam    290 non-null    str     
 1   points  290 non-null    Int64   
 2   basket  290 non-null    category
dtypes: Int64(1), category(1), str(1)
memory usage: 5.3 KB


In [10]:
res = data.groupby(by=['exam', 'basket']).count()
res

points
exam  basket        
exam1 NA           7
      mark2        5
      mark3       18
      mark4       20
      mark5       18
exam2 NA           2
      mark2        7
      mark3       24
      mark4       24
      mark5       30
exam3 NA          18
      mark2        3
      mark3       57
      mark4       41
      mark5       16

In [11]:
stat = data.pivot_table(index='exam', columns='basket', 
                        values='points', aggfunc='count')
stat

basket,NA,mark2,mark3,mark4,mark5
exam,,,,,
exam1,7,5,18,20,18
exam2,2,7,24,24,30
exam3,18,3,57,41,16


In [12]:
stat.columns

CategoricalIndex(['NA', 'mark2', 'mark3', 'mark4', 'mark5'], categories=['NA', 'mark2', 'mark3', 'mark4', 'mark5'], ordered=True, dtype='category', name='basket')

In [13]:
stat['total'] = stat.sum(axis=1)
stat

basket,NA,mark2,mark3,mark4,mark5,total
exam,,,,,,
exam1,7,5,18,20,18,68
exam2,2,7,24,24,30,87
exam3,18,3,57,41,16,135


In [14]:
t = stat.copy()
t.NA /= t.total
t.mark2 /= t.total
t.mark3 /= t.total
t.mark4 /= t.total
t.mark5 /= t.total
t.drop(columns='total', inplace=True)
t

basket,NA,mark2,mark3,mark4,mark5
exam,,,,,
exam1,0.102941,0.073529,0.264706,0.294118,0.264706
exam2,0.022989,0.08046,0.275862,0.275862,0.344828
exam3,0.133333,0.022222,0.422222,0.303704,0.118519


In [15]:
def my_f(x):
    return f'{x:.1%}'
t.map(my_f)

basket,NA,mark2,mark3,mark4,mark5
exam,,,,,
exam1,10.3%,7.4%,26.5%,29.4%,26.5%
exam2,2.3%,8.0%,27.6%,27.6%,34.5%
exam3,13.3%,2.2%,42.2%,30.4%,11.9%
